# Colab-Optimized Docling PDF Ingestion
This notebook is specifically designed to run flawlessly on Google Colab.
- It uses Google Drive for storage.
- It uses `ThreadPoolExecutor` instead of `ProcessPoolExecutor` to avoid Colab RAM exhaustion and GPU multiprocessing deadlocks.
- Everything is self-contained in a single notebook file.

In [ ]:
# 1. Install dependencies (Required for Colab)
!pip install -q docling psutil pandas seaborn matplotlib

In [ ]:
# 2. Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# 3. Configuration & Options
import os
from pathlib import Path

# --- PATHS ---
# Update these paths to point to your specific folders in Google Drive
SOURCE_DIR = Path("/content/drive/MyDrive/nlp_ml_gcs_pdfs")
OUTPUT_DIR = Path("/content/drive/MyDrive/docling_parsed_output")

# --- OPTIONS ---
MAX_FILES = 100       # Max files to process (Set to None to process all)
MAX_THREADS = 2       # Number of concurrent threads (1-4 recommended for Colab)
DISABLE_OCR = True    # Set to True to disable OCR (much faster), False to enable

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print(f"Reading PDFs from: {SOURCE_DIR}")
print(f"Writing JSONs to:  {OUTPUT_DIR}")

In [ ]:
# 4. Initialize Global Docling Model
import torch
from docling.document_converter import DocumentConverter, PdfFormatOption
from docling.datamodel.base_models import InputFormat
from docling.datamodel.pipeline_options import PdfPipelineOptions

print("Initializing Docling Model (This may take a moment to download weights)...")
pipeline_options = PdfPipelineOptions()
pipeline_options.do_ocr = not DISABLE_OCR
pipeline_options.do_table_structure = True

# We create a single global converter. 
# ThreadPoolExecutor will share this instance, keeping memory usage low!
converter = DocumentConverter(
    allowed_formats=[InputFormat.PDF],
    format_options={
        InputFormat.PDF: PdfFormatOption(
            pipeline_options=pipeline_options,
        )
    }
)
print("Model Initialized successfully!")

In [ ]:
# 5. Define Worker Function
import time
import json

def process_single_pdf(pdf_path: Path, output_dir: Path):
    file_size_kb = pdf_path.stat().st_size / 1024
    start_time = time.perf_counter()
    status = "SUCCESS"
    error_msg = ""
    
    try:
        # Convert using the shared global model
        result = converter.convert(pdf_path)
        
        # Extract dict
        doc_dict = result.document.export_to_dict()
        out_path = output_dir / f"{pdf_path.stem}.json"
        
        # Write to Google Drive
        with open(out_path, "w") as f:
            json.dump(doc_dict, f)
            
    except Exception as e:
        status = "FAILED"
        error_msg = str(e)
        
    end_time = time.perf_counter()
    processing_time = end_time - start_time
    
    return {
        "filename": pdf_path.name,
        "file_size_kb": file_size_kb,
        "processing_time_sec": processing_time,
        "status": status,
        "error_msg": error_msg
    }

In [ ]:
# 6. Run Multithreaded Pipeline
from concurrent.futures import ThreadPoolExecutor, as_completed
from tqdm.auto import tqdm
import pandas as pd

def run_ingestion_pipeline():
    # Gather files
    pdf_files = list(SOURCE_DIR.glob("*.pdf"))
    if MAX_FILES is not None:
        pdf_files = pdf_files[:MAX_FILES]
        
    print(f"Found {len(pdf_files)} PDFs to process.")
    if len(pdf_files) == 0:
        return pd.DataFrame(), 0.0

    metrics = []
    start_total = time.perf_counter()
    
    # ThreadPoolExecutor bypasses Notebook multiprocessing issues
    # It's completely safe on Colab and uses the C/C++ backend of PyTorch for parallelization.
    with ThreadPoolExecutor(max_workers=MAX_THREADS) as executor:
        futures = {executor.submit(process_single_pdf, pdf, OUTPUT_DIR): pdf for pdf in pdf_files}
        
        for future in tqdm(as_completed(futures), total=len(futures), desc="Parsing PDFs"):
            metrics.append(future.result())
            
    end_total = time.perf_counter()
    total_time = end_total - start_total
    
    df_metrics = pd.DataFrame(metrics)
    print(f"\nPipeline completed in {total_time:.2f} seconds.")
    
    return df_metrics, total_time

df_metrics, total_time = run_ingestion_pipeline()

In [ ]:
# 7. Analytics and Metrics
import seaborn as sns
import matplotlib.pyplot as plt

if not df_metrics.empty:
    success_rate = (df_metrics['status'] == 'SUCCESS').mean() * 100
    avg_time = df_metrics.loc[df_metrics['status'] == 'SUCCESS', 'processing_time_sec'].mean()
    throughput = len(df_metrics) / total_time

    print(f"=== INGESTION SUMMARY ===")
    print(f"Total PDFs Processed: {len(df_metrics)}")
    print(f"Success Rate:         {success_rate:.2f}%")
    print(f"Total Wall Time:      {total_time:.2f} s")
    print(f"Throughput:           {throughput:.2f} PDFs / sec")
    print(f"Avg Time per PDF:     {avg_time:.2f} s")

    failures = df_metrics[df_metrics['status'] == 'FAILED']
    if not failures.empty:
        print(f"\nEncountered {len(failures)} failures:")
        display(failures[['filename', 'error_msg']].head())

    plt.figure(figsize=(10, 5))
    sns.histplot(data=df_metrics[df_metrics['status'] == 'SUCCESS'], x='processing_time_sec', bins=30, kde=True)
    plt.title('Distribution of Docling Processing Times per PDF')
    plt.xlabel('Processing Time (seconds)')
    plt.ylabel('Count')
    plt.show()
else:
    print("No data to analyze.")